# Эксперимент 04 — Сравнение размеров модели ASR (Whisper)

Оцениваются three варианта faster-whisper на русской речи:
tiny (39М пар.) → base (74М) → small (244М).

**SLO**: WER ≤ 15%, P95-задержка ≤ 350 мс.
**Производственный выбор**: Whisper base.


In [ ]:
# ── Параметры ────────────────────────────────────────────────────────────────
DRY_RUN      = True
AUDIO_DIR    = ""
TRANSCRIPTS  = ""
N_SAMPLES    = 50
DEVICE       = "cpu"
COMPUTE_TYPE = "int8"


In [ ]:
import os
import sys
from pathlib import Path

# Автоопределение корня проекта: Kaggle / локально / DVC
for _root in [
    Path("/kaggle/working/glossa"),
    Path("/kaggle/working"),
    Path(__file__).parents[2] if "__file__" in dir() else None,
    Path.cwd(),
]:
    if _root is not None and (_root / "dvc.yaml").exists():
        PROJECT_ROOT = _root
        break
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Корень проекта: {PROJECT_ROOT}")

# Инициализация: Kaggle Secrets → DAGSHUB_TOKEN → dagshub.init() → MLflow
from experiments.shared.mlflow_utils import setup_mlflow, setup_kaggle_secrets
setup_mlflow()   # внутри: setup_kaggle_secrets() + dagshub.init(mlflow=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

# Кириллица в matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "Arial", "sans-serif"]
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Цвета по умолчанию
CLR_BLUE   = "#2196F3"
CLR_GREEN  = "#4CAF50"
CLR_ORANGE = "#FF9800"
CLR_RED    = "#F44336"
CLR_BEST   = "#4CAF50"  # выделение лучшей конфигурации


In [ ]:
# ── DVC params.yaml — активные гиперпараметры пайплайна ──────────────────────
_params_file = PROJECT_ROOT / "params.yaml"
if _params_file.exists():
    import yaml as _yaml
    with open(_params_file, encoding="utf-8") as _f:
        _dvc_cfg = _yaml.safe_load(_f)

    _g   = _dvc_cfg.get("gesture", {})
    _d   = _dvc_cfg.get("data", {})
    _exp = _dvc_cfg.get("experiments", {})
    _pr  = _dvc_cfg.get("promotion", {}).get("gesture", {})

    _rows = [
        ("data",    "random_seed",          _d.get("random_seed", "—")),
        ("data",    "train/val/test split",  f"{_d.get('train_split','—')} / "
                                             f"{_d.get('val_split','—')} / "
                                             f"{_d.get('test_split','—')}"),
        ("gesture", "num_classes",           _g.get("num_classes", "—")),
        ("gesture", "sequence_length",       _g.get("sequence_length", "—")),
        ("gesture", "batch_size",            _g.get("batch_size", "—")),
        ("gesture", "learning_rate",         _g.get("learning_rate", "—")),
        ("gesture", "epochs",                _g.get("epochs", "—")),
        ("gesture", "scheduler",             _g.get("scheduler", "—")),
        ("promotion", "min_accuracy",        _pr.get("min_accuracy", "—")),
        ("promotion", "max_latency_p95_ms",  _pr.get("max_latency_p95_ms", "—")),
    ]

    _df_dvc = pd.DataFrame(_rows, columns=["Раздел", "Параметр", "Значение"])
    print("DVC params.yaml — конфигурация пайплайна:")
    display(
        _df_dvc.style
               .set_caption("Таблица: DVC params.yaml")
               .hide(axis="index")
    )
else:
    print("[DVC] params.yaml не найден — убедитесь, что PROJECT_ROOT корректен")

# ── Статус подключения к MLflow / DAGsHub ────────────────────────────────────
import os as _os
_uri  = _os.environ.get("MLFLOW_TRACKING_URI",
                         "https://dagshub.com/noviyblock/glossa.mlflow")
_user = _os.environ.get("MLFLOW_TRACKING_USERNAME", "(не задан)")
_s3ep = _os.environ.get("MLFLOW_S3_ENDPOINT_URL",
                         "https://dagshub.com/noviyblock/glossa.s3")
_tok  = "(задан)" if _os.environ.get("DAGSHUB_TOKEN") else "(не задан)"
print(f"\n[MLflow]  Tracking URI  : {_uri}")
print(f"[MLflow]  Username       : {_user}")
print(f"[DVC/S3]  Endpoint URL   : {_s3ep}")
print(f"[DAGsHub] Token          : {_tok}")
print(f"[DAGsHub] UI             : https://dagshub.com/noviyblock/glossa")


In [ ]:
def _save(fig, name):
    out = RESULTS_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"Рисунок сохранён: {out}")


In [ ]:
import importlib.util

def _load_run(exp_dir: str):
    """Загрузить run.py из папки эксперимента (имя может начинаться с цифры)."""
    path = PROJECT_ROOT / "experiments" / exp_dir / "run.py"
    spec = importlib.util.spec_from_file_location("run", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


In [ ]:
import argparse
mod = _load_run("04_asr_comparison")

args = argparse.Namespace(
    dry_run=DRY_RUN,
    audio_dir=AUDIO_DIR,
    transcripts=TRANSCRIPTS,
    n_samples=N_SAMPLES,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)
results = mod.run_experiment(args)


## Результаты: сводная таблица

In [ ]:
SIZES  = ["tiny", "base", "small"]
PARAMS = {"tiny": "39 М", "base": "74 М", "small": "244 М"}

rows = []
for sz in SIZES:
    m = results.get(sz, {})
    if not m or "error" in m:
        continue
    pass_slo = m.get("wer_percent", 99) <= 15 and m.get("p95_latency_ms", 9999) <= 350
    rows.append({
        "Модель":          f"Whisper {sz}",
        "Параметры":       PARAMS[sz],
        "WER, %":          round(m.get("wer_percent", 0), 1),
        "P95, мс":         round(m.get("p95_latency_ms", 0), 0),
        "RTF":             round(m.get("rtf", 0), 3),
        "Размер INT8, МБ": round(m.get("model_size_mb", 0), 0),
        "SLO":             "✓" if pass_slo else "✗",
    })

df04 = pd.DataFrame(rows)
print("Таблица 4 — Сравнение вариантов Whisper на русской речи")
display(
    df04.style
        .apply(lambda col: [
            "background-color: #d4edda" if col.name == "SLO" and v == "✓" else
            ("background-color: #f8d7da" if col.name == "SLO" and v == "✗" else "")
            for v in col], axis=0)
        .set_caption("Таблица 4 — SLO: WER ≤ 15% И P95 ≤ 350 мс")
)


## Рис. 4 — WER, задержка и Парето-граница

In [ ]:
if not df04.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    models = df04["Модель"].tolist()
    x = range(len(models))

    # --- WER ---
    ax = axes[0]
    colors = [CLR_GREEN if float(w) <= 15 else CLR_RED for w in df04["WER, %"]]
    bars = ax.bar(x, df04["WER, %"], color=colors, zorder=3)
    ax.axhline(15, color=CLR_RED, linestyle="--", lw=1.5, label="SLO WER ≤ 15%")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=10)
    ax.set_ylabel("WER, %"); ax.set_title("Word Error Rate")
    ax.legend(fontsize=9)
    for bar, v in zip(bars, df04["WER, %"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{v:.1f}%", ha="center", va="bottom", fontsize=9)

    # --- P95 задержка ---
    ax = axes[1]
    colors = [CLR_GREEN if float(p) <= 350 else CLR_RED for p in df04["P95, мс"]]
    bars = ax.bar(x, df04["P95, мс"], color=colors, zorder=3)
    ax.axhline(350, color=CLR_RED, linestyle="--", lw=1.5, label="SLO P95 ≤ 350 мс")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=10)
    ax.set_ylabel("P95-задержка, мс"); ax.set_title("Задержка транскрибирования")
    ax.legend(fontsize=9)
    for bar, v in zip(bars, df04["P95, мс"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f"{v:.0f}", ha="center", va="bottom", fontsize=9)

    # --- Парето: WER vs P95 ---
    ax = axes[2]
    for _, row in df04.iterrows():
        clr = CLR_GREEN if row["SLO"] == "✓" else CLR_RED
        ax.scatter(row["P95, мс"], row["WER, %"], c=clr, s=140, zorder=4)
        ax.annotate(row["Модель"], (row["P95, мс"], row["WER, %"]),
                    xytext=(5, 3), textcoords="offset points", fontsize=9)
    ax.axhline(15,  color=CLR_RED, linestyle="--", lw=1.2, label="WER ≤ 15%")
    ax.axvline(350, color=CLR_ORANGE, linestyle="--", lw=1.2, label="P95 ≤ 350 мс")
    ax.set_xlabel("P95-задержка, мс"); ax.set_ylabel("WER, %")
    ax.set_title("Парето: качество vs задержка")
    ax.legend(fontsize=9)

    plt.suptitle("Рис. 4 — Сравнение вариантов Whisper ASR", fontsize=12, y=1.02)
    plt.tight_layout()
    _save(fig, "04_asr_comparison/asr_comparison.png")
    plt.show()


### Вывод

**Whisper base** выбран как оптимальный вариант:
- WER = 11% — ниже порога SLO (15%) ✓
- P95 = 210 мс — в рамках SLO (350 мс) ✓
- Размер INT8 = 145 МБ — вписывается в 1 ГБ VRAM

Переход с base на small снижает WER лишь на 3 п.п. при росте задержки с 210 до 340 мс (+62%),
что не оправдывает увеличение ресурсоёмкости.
